In [ ]:
# import os
# import sys
# project_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
# sys.path.append(project_path)
#
# requirements_path = os.path.join(project_path, "SECONDARY/requirements.txt")
# !{sys.executable} -m pip install -r "{requirements_path}"

In [ ]:
import os
import sys
import time
!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [ ]:
%load_ext autoreload
%autoreload

from datetime import datetime
from datetime import timedelta
from dateutil import parser
from SRC.LIBRARIES.new_data_utils import fetch
from SRC.CORE._CONSTANTS import _KIEV_TIMESTAMP
import SRC.LIBRARIES.new_fibonacci_plot_utils as nfpu
import SRC.LIBRARIES.new_fibonacci_statistics_utils as nfsu
import SRC.LIBRARIES.new_utils as nu

# todo xrp 4h 2025 and 2026 champion.
# todo doge/xrp 4h 2026 champions
sym='xrp'.upper()
symbol = f'{sym}USDT'
discretization = '4h'.upper()
display_start_date_str = '2026-01-01'
display_end_date_str = '2026-05-21'
load_end_date = parser.parse(display_end_date_str)
capital_per_trade = 1000
commission_rate = 0.00075

market_type = 'SPOT'
mrc_buffer = 1000
rsi_buffer = 200
stoch_buffer = 50
macd_buffer = 200
atr_buffer = 200
display_start_dt = parser.parse(display_start_date_str)
start_time = time.perf_counter()

discretization_value = int(discretization[:-1])
timeframe_seconds = {
    'M': discretization_value * 60,
    'H': discretization_value * 3600,
    'D': discretization_value * 86400
}.get(discretization[-1], 1800)

load_start_dt = display_start_dt - timedelta(seconds=timeframe_seconds * mrc_buffer)

mrc_df = fetch(market_type, symbol, discretization, load_start_dt, load_end_date)
df = mrc_df.iloc[mrc_buffer:].copy()

In [ ]:
# Убеждаемся, что индексы уникальны
mrc_df = mrc_df[~mrc_df.index.duplicated(keep='first')]
df = df[~df.index.duplicated(keep='first')]

# 1. RSI
rsi_calc_df = nu.prepare_buffer_data(mrc_df, df, rsi_buffer)
df = nu.rsi(df, rsi_calc_df)

# 2. Stochastic
stoch_calc_df = nu.prepare_buffer_data(mrc_df, df, stoch_buffer)
df = nu.stochastic_tradingview(df, stoch_calc_df)

# 3. MACD
macd_calc_df = nu.prepare_buffer_data(mrc_df, df, macd_buffer)
df, macd = nu.macd(df, macd_calc_df)

# 4. ATR
atr_calc_df = nu.prepare_buffer_data(mrc_df, df, atr_buffer)
df = nu.atr(df, atr_calc_df)

# Расчёт MRC
mrc_df = nu.mrc_calculate(mrc_df, df)

# SMA для фильтра (период можно менять)
sma_period = 20
df['sma'] = df['close'].rolling(sma_period).mean()

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Tuple

def backtest_mrc_stochastic_with_trailing(df: pd.DataFrame,
                                           commission: float = 0.00075,
                                           capital_per_trade: float = 1000.0,
                                           slippage: float = 0.0002,
                                           lookback_min_bars: int = 8,
                                           slope_threshold: float = 0.0,
                                           stoch_oversold: float = 30.0,
                                           stoch_overbought: float = 70.0,
                                           require_retest: bool = False,
                                           retest_max_bars: int = 5,
                                           use_sma_filter: bool = False,
                                           use_slope_filter: bool = True,
                                           use_stoch_filter: bool = True,
                                           only_long: bool = False,
                                           only_short: bool = False,
                                           trailing_sl: bool = False,
                                           trailing_sl_distance: float = 0.5,
                                           trailing_tp: bool = False,
                                           trailing_tp_distance: float = 1.0,
                                           sl_mode: str = 'mrc_band',
                                           sl_ratio: float = 3.0
                                          ) -> Tuple[pd.DataFrame, Dict]:
    """
    Гибкая комбинированная стратегия MRC + Стохастик с трейлингами.
    - Трейлинг-стоп: подтягивает стоп-лосс за ценой, но не ниже первоначального SL
    - Трейлинг-ТП: подтягивает тейк-профит за ценой, но не хуже первоначального TP
    - sl_mode: 'mrc_band' - стоп на красной линии, 'fixed_ratio' - фиксированное соотношение риск/прибыль
    - sl_ratio: соотношение риск:прибыль (например, 3.0 означает риск 1, прибыль 3)
    """
    df_back = df.copy()

    # Наклон meanline
    df_back['slope'] = df_back['meanline'] - df_back['meanline'].shift(1)
    df_back['slope_dir'] = 'neutral'
    df_back.loc[df_back['slope'] > slope_threshold, 'slope_dir'] = 'up'
    df_back.loc[df_back['slope'] < -slope_threshold, 'slope_dir'] = 'down'

    # Касание зелёных полос
    long_touch = (df_back['low'] <= df_back['loband1']) & (df_back['high'] >= df_back['loband1'])
    short_touch = (df_back['high'] >= df_back['upband1']) & (df_back['low'] <= df_back['upband1'])

    # Базовые сигналы
    long_base = long_touch
    short_base = short_touch

    # Фильтр по стохастику
    if use_stoch_filter:
        stoch_long_zone = df_back['stoch_k'] < stoch_oversold
        stoch_short_zone = df_back['stoch_k'] > stoch_overbought
        long_base = long_base & stoch_long_zone
        short_base = short_base & stoch_short_zone

    # Фильтр по наклону
    if use_slope_filter:
        long_base = long_base & (df_back['slope_dir'] == 'up')
        short_base = short_base & (df_back['slope_dir'] == 'down')

    # SMA фильтр
    if use_sma_filter and 'sma' in df_back.columns:
        sma_condition_long = df_back['close'] > df_back['sma']
        sma_condition_short = df_back['close'] < df_back['sma']
        sma_condition_long = sma_condition_long.fillna(True)
        sma_condition_short = sma_condition_short.fillna(True)
        long_base = long_base & sma_condition_long
        short_base = short_base & sma_condition_short

    # Ограничения по направлению
    if only_long:
        short_base = pd.Series(False, index=df_back.index)
    if only_short:
        long_base = pd.Series(False, index=df_back.index)

    # Ретест
    if require_retest:
        long_waiting = False
        long_first_touch_idx = -1
        short_waiting = False
        short_first_touch_idx = -1

    df_back['signal'] = 0
    df_back['entry_price'] = np.nan
    df_back['exit_price'] = np.nan
    df_back['pnl'] = 0.0
    df_back['trade_type'] = ''

    position = None
    trades = []

    for i in range(1, len(df_back)):
        current_long_signal = long_base.iloc[i]
        current_short_signal = short_base.iloc[i]

        # Ретест
        if require_retest:
            if long_waiting and (i - long_first_touch_idx) > retest_max_bars:
                long_waiting = False
                long_first_touch_idx = -1
            if short_waiting and (i - short_first_touch_idx) > retest_max_bars:
                short_waiting = False
                short_first_touch_idx = -1

            if not long_waiting and long_touch.iloc[i]:
                long_waiting = True
                long_first_touch_idx = i
            if not short_waiting and short_touch.iloc[i]:
                short_waiting = True
                short_first_touch_idx = i

            long_retest_ok = False
            short_retest_ok = False
            if long_waiting and long_touch.iloc[i] and i > long_first_touch_idx:
                if i - long_first_touch_idx >= 2:
                    check_idx = long_first_touch_idx + 1
                    if check_idx < len(df_back) and df_back.iloc[check_idx]['close'] > df_back.iloc[long_first_touch_idx]['loband1']:
                        long_retest_ok = True
                        long_waiting = False
                        long_first_touch_idx = -1
            if short_waiting and short_touch.iloc[i] and i > short_first_touch_idx:
                if i - short_first_touch_idx >= 2:
                    check_idx = short_first_touch_idx + 1
                    if check_idx < len(df_back) and df_back.iloc[check_idx]['close'] < df_back.iloc[short_first_touch_idx]['upband1']:
                        short_retest_ok = True
                        short_waiting = False
                        short_first_touch_idx = -1

            current_long_signal = current_long_signal and long_retest_ok
            current_short_signal = current_short_signal and short_retest_ok

        # Вход
        if position is None:
            can_enter = True
            if len(trades) > 0 and (i - trades[-1]['exit_idx']) < lookback_min_bars:
                can_enter = False

            if can_enter:
                if current_long_signal:
                    raw_entry = df_back.iloc[i]['loband1']
                    entry_price = raw_entry * (1 + slippage)
                    amount = (capital_per_trade * (1 - commission)) / entry_price
                    tp_price = df_back.iloc[i]['upband1']
                    
                    # Расчёт SL в зависимости от режима
                    if sl_mode == 'mrc_band':
                        sl_price = df_back.iloc[i]['loband2']
                    elif sl_mode == 'fixed_ratio':
                        distance_to_tp = tp_price - entry_price
                        sl_price = entry_price - distance_to_tp / sl_ratio
                    else:
                        sl_price = df_back.iloc[i]['loband2']
                    
                    position = {
                        'type': 'long',
                        'entry_idx': i,
                        'entry_price': entry_price,
                        'amount': amount,
                        'tp': tp_price,
                        'initial_tp': tp_price,
                        'sl': sl_price,
                        'initial_sl': sl_price,
                        'sl_trailing_active': trailing_sl,
                        'sl_highest_price': entry_price,
                        'tp_trailing_active': trailing_tp,
                        'tp_highest_price': entry_price,
                    }
                    df_back.at[df_back.index[i], 'signal'] = 1
                    df_back.at[df_back.index[i], 'entry_price'] = entry_price
                    continue

                if current_short_signal:
                    raw_entry = df_back.iloc[i]['upband1']
                    entry_price = raw_entry * (1 - slippage)
                    amount = (capital_per_trade * (1 - commission)) / entry_price
                    tp_price = df_back.iloc[i]['loband1']
                    
                    # Расчёт SL в зависимости от режима
                    if sl_mode == 'mrc_band':
                        sl_price = df_back.iloc[i]['upband2']
                    elif sl_mode == 'fixed_ratio':
                        distance_to_tp = entry_price - tp_price
                        sl_price = entry_price + distance_to_tp / sl_ratio
                    else:
                        sl_price = df_back.iloc[i]['upband2']
                    
                    position = {
                        'type': 'short',
                        'entry_idx': i,
                        'entry_price': entry_price,
                        'amount': amount,
                        'tp': tp_price,
                        'initial_tp': tp_price,
                        'sl': sl_price,
                        'initial_sl': sl_price,
                        'sl_trailing_active': trailing_sl,
                        'sl_lowest_price': entry_price,
                        'tp_trailing_active': trailing_tp,
                        'tp_lowest_price': entry_price,
                    }
                    df_back.at[df_back.index[i], 'signal'] = -1
                    df_back.at[df_back.index[i], 'entry_price'] = entry_price
                    continue

        # Выход
        else:
            current_high = df_back.iloc[i]['high']
            current_low = df_back.iloc[i]['low']
            exit_price = None
            exit_type = None

            if position['type'] == 'long':
                # Обновление максимумов
                if current_high > position['sl_highest_price']:
                    position['sl_highest_price'] = current_high
                if current_high > position['tp_highest_price']:
                    position['tp_highest_price'] = current_high

                # Обновление трейлинг-стопа (только вверх, не ниже initial_sl)
                if position['sl_trailing_active']:
                    new_sl = position['sl_highest_price'] * (1 - trailing_sl_distance / 100)
                    if new_sl > position['initial_sl']:
                        position['sl'] = max(position['sl'], new_sl)

                # Обновление трейлинг-ТП (только вверх, не ниже initial_tp)
                if position['tp_trailing_active']:
                    new_tp = position['tp_highest_price'] * (1 - trailing_tp_distance / 100)
                    if new_tp > position['initial_tp']:
                        position['tp'] = max(position['tp'], new_tp)

                # Проверка выходов
                if current_high >= position['tp']:
                    exit_price = position['tp']
                    exit_type = 'tp_trailing' if position['tp_trailing_active'] else 'tp'
                elif current_low <= position['sl']:
                    exit_price = position['sl']
                    exit_type = 'sl_trailing' if position['sl_trailing_active'] else 'sl'

            else:  # short
                # Обновление минимумов
                if current_low < position['sl_lowest_price']:
                    position['sl_lowest_price'] = current_low
                if current_low < position['tp_lowest_price']:
                    position['tp_lowest_price'] = current_low

                # Обновление трейлинг-стопа (только вниз, не выше initial_sl)
                if position['sl_trailing_active']:
                    new_sl = position['sl_lowest_price'] * (1 + trailing_sl_distance / 100)
                    if new_sl < position['initial_sl']:
                        position['sl'] = min(position['sl'], new_sl)

                # Обновление трейлинг-ТП (только вниз, не выше initial_tp)
                if position['tp_trailing_active']:
                    new_tp = position['tp_lowest_price'] * (1 + trailing_tp_distance / 100)
                    if new_tp < position['initial_tp']:
                        position['tp'] = min(position['tp'], new_tp)

                # Проверка выходов
                if current_low <= position['tp']:
                    exit_price = position['tp']
                    exit_type = 'tp_trailing' if position['tp_trailing_active'] else 'tp'
                elif current_high >= position['sl']:
                    exit_price = position['sl']
                    exit_type = 'sl_trailing' if position['sl_trailing_active'] else 'sl'

            if exit_price is not None:
                if position['type'] == 'long':
                    exit_price_adj = exit_price * (1 - slippage)
                    exit_proceeds = position['amount'] * exit_price_adj * (1 - commission)
                    pnl = exit_proceeds - capital_per_trade
                else:
                    exit_price_adj = exit_price * (1 + slippage)
                    entry_proceeds = position['amount'] * position['entry_price'] * (1 - commission)
                    exit_cost = position['amount'] * exit_price_adj * (1 + commission)
                    pnl = entry_proceeds - exit_cost

                trades.append({
                    'entry_idx': position['entry_idx'],
                    'exit_idx': i,
                    'type': position['type'],
                    'entry_price': position['entry_price'],
                    'exit_price': exit_price_adj,
                    'pnl': pnl,
                    'exit_type': exit_type
                })
                df_back.at[df_back.index[i], 'exit_price'] = exit_price_adj
                df_back.at[df_back.index[i], 'pnl'] = pnl
                df_back.at[df_back.index[i], 'trade_type'] = f"{position['type']}_{exit_type}"
                position = None

    # Принудительное закрытие
    if position is not None:
        last_price = df_back.iloc[-1]['close']
        if position['type'] == 'long':
            exit_price_adj = last_price * (1 - slippage)
            exit_proceeds = position['amount'] * exit_price_adj * (1 - commission)
            pnl = exit_proceeds - capital_per_trade
        else:
            exit_price_adj = last_price * (1 + slippage)
            entry_proceeds = position['amount'] * position['entry_price'] * (1 - commission)
            exit_cost = position['amount'] * exit_price_adj * (1 + commission)
            pnl = entry_proceeds - exit_cost
        trades.append({
            'entry_idx': position['entry_idx'],
            'exit_idx': len(df_back)-1,
            'type': position['type'],
            'entry_price': position['entry_price'],
            'exit_price': exit_price_adj,
            'pnl': pnl,
            'exit_type': 'forced'
        })
        df_back.at[df_back.index[-1], 'exit_price'] = exit_price_adj
        df_back.at[df_back.index[-1], 'pnl'] = pnl
        df_back.at[df_back.index[-1], 'trade_type'] = f"{position['type']}_forced"

    # Метрики
    trades_df = pd.DataFrame(trades)
    if len(trades_df) == 0:
        print("Нет сделок.")
        return df_back, {'total_pnl': 0, 'win_rate': 0, 'num_trades': 0, 'avg_win': 0, 'avg_loss': 0, 'max_drawdown': 0, 'sharpe_approx': 0}

    total_pnl = trades_df['pnl'].sum()
    avg_return_per_trade_percent = (total_pnl / (capital_per_trade * len(trades_df))) * 100
    win_rate = (trades_df['pnl'] > 0).mean()
    num_trades = len(trades_df)
    avg_win = trades_df[trades_df['pnl'] > 0]['pnl'].mean() if (trades_df['pnl'] > 0).any() else 0.0
    avg_loss = trades_df[trades_df['pnl'] < 0]['pnl'].mean() if (trades_df['pnl'] < 0).any() else 0.0
    cum_pnl = trades_df['pnl'].cumsum()
    running_max = cum_pnl.cummax()
    max_drawdown = (running_max - cum_pnl).max()
    returns = trades_df['pnl'] / capital_per_trade
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if len(returns) > 1 and returns.std() != 0 else 0.0

    metrics = {
        'total_pnl': total_pnl,
        'avg_return_per_trade_percent': avg_return_per_trade_percent,
        'win_rate': win_rate,
        'num_trades': num_trades,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'max_drawdown': max_drawdown,
        'sharpe_approx': sharpe
    }
    return df_back, metrics

In [ ]:
%load_ext autoreload
%autoreload

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import SRC.LIBRARIES.new_indicator_plot_utils as nipu

# ========== НАСТРОЙКИ ==========
# Основные фильтры
USE_STOCH_FILTER = True
USE_SLOPE_FILTER = True
USE_SMA_FILTER = False
REQUIRE_RETEST = False

# Ограничения по направлению
ONLY_LONG = False
ONLY_SHORT = False

# Параметры стохастика и MRC
SLOPE_THRESHOLD = 0.0
STOCH_OVERSOLD = 30.0
STOCH_OVERBOUGHT = 70.0
RETEST_MAX_BARS = 5
SMA_PERIOD = 20

# ========== НАСТРОЙКИ СТОП-ЛОССА ==========
SL_MODE = 'mrc_band'      # 'mrc_band' - красная линия, 'fixed_ratio' - фиксированное соотношение
SL_RATIO = 3.0               # соотношение риск:прибыль (2.0 = риск 1, прибыль 2)

# ========== ТРЕЙЛИНГИ ==========
TRAILING_SL = False                      # включить трейлинг-стоп
TRAILING_SL_DISTANCE = 15.0              # % отступа от максимума
TRAILING_TP = True                       # включить трейлинг-тейк-профит
TRAILING_TP_DISTANCE = 0.5               # % отступа от максимума
# ==================================

# Обновляем SMA
df['sma'] = df['close'].rolling(SMA_PERIOD).mean()

is_log_scale_by_default = True
candlestick_row = 1

fig_main = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.03)

fig_main.add_trace(
    go.Candlestick(
        x=df[_KIEV_TIMESTAMP],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC"
    ),
    row=candlestick_row, col=1
)

nipu.add_mrc(candlestick_row, fig_main, df)

# Запуск бэктеста
df_backtest, metrics = backtest_mrc_stochastic_with_trailing(
    df,
    commission=commission_rate,
    capital_per_trade=capital_per_trade,
    slippage=0.0002,
    lookback_min_bars=8,
    slope_threshold=SLOPE_THRESHOLD,
    stoch_oversold=STOCH_OVERSOLD,
    stoch_overbought=STOCH_OVERBOUGHT,
    require_retest=REQUIRE_RETEST,
    retest_max_bars=RETEST_MAX_BARS,
    use_sma_filter=USE_SMA_FILTER,
    use_slope_filter=USE_SLOPE_FILTER,
    use_stoch_filter=USE_STOCH_FILTER,
    only_long=ONLY_LONG,
    only_short=ONLY_SHORT,
    trailing_sl=TRAILING_SL,
    trailing_sl_distance=TRAILING_SL_DISTANCE,
    trailing_tp=TRAILING_TP,
    trailing_tp_distance=TRAILING_TP_DISTANCE,
    sl_mode=SL_MODE,
    sl_ratio=SL_RATIO
)

print(f"Результаты комбинированной стратегии MRC+Stoch с трейлингами:")
print(f"{sym} {discretization} | {display_start_date_str} - {display_end_date_str}")
print(f"Stoch: STOCH_OVERSOLD={STOCH_OVERSOLD}, STOCH_OVERBOUGHT={STOCH_OVERBOUGHT}")
print(f"Фильтры: Stoch={USE_STOCH_FILTER}, Slope={USE_SLOPE_FILTER}, SMA={USE_SMA_FILTER}")
print(f"Режим SL: {SL_MODE}")
if SL_MODE == 'fixed_ratio':
    print(f"Соотношение риск:прибыль = 1:{SL_RATIO:.1f}")
print(f"Трейлинг-стоп: включён={TRAILING_SL}, отступ={TRAILING_SL_DISTANCE}%")
print(f"Трейлинг-ТП: включён={TRAILING_TP}, отступ={TRAILING_TP_DISTANCE}%")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

price_log_scale_value = "log"
price_linear_scale_value = "linear"
price_log_scale_title = "Price (log scale)"
price_linear_scale_title = "Price (linear scale)"

fig_main.update_layout(
    title=f"🐵{discretization} MRC+Stoch | SL mode={SL_MODE}",
    xaxis_rangeslider_visible=False,
    yaxis1_type=price_log_scale_value if is_log_scale_by_default else price_linear_scale_value,
    yaxis1_title=price_log_scale_title if is_log_scale_by_default else price_linear_scale_title,
    height=1200,
    bargap=0,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=1 if is_log_scale_by_default else 0,
            x=0.115,
            y=1.073,
            buttons=[
                dict(label="Linear", method="relayout", args=[{"yaxis.type": price_linear_scale_value, "yaxis.title.text": price_linear_scale_title}]),
                dict(label="Log", method="relayout", args=[{"yaxis.type": price_log_scale_value, "yaxis.title.text": price_log_scale_title}])
            ],
            font=dict(color="red", size=12, family="Arial"),
            bgcolor="rgba(0, 0, 0, 0)",
            bordercolor="red",
            borderwidth=1
        )
    ]
)

# Маркеры
entries_long = df_backtest[df_backtest['signal'] == 1].index
entries_short = df_backtest[df_backtest['signal'] == -1].index
exits_tp = df_backtest[df_backtest['trade_type'].str.contains('tp', na=False) & ~df_backtest['trade_type'].str.contains('trailing', na=False)].index
exits_tp_trailing = df_backtest[df_backtest['trade_type'].str.contains('tp_trailing', na=False)].index
exits_sl = df_backtest[df_backtest['trade_type'].str.contains('sl', na=False) & ~df_backtest['trade_type'].str.contains('trailing', na=False)].index
exits_sl_trailing = df_backtest[df_backtest['trade_type'].str.contains('sl_trailing', na=False)].index

fig_main.add_trace(go.Scatter(x=df_backtest.loc[entries_long, _KIEV_TIMESTAMP], y=df_backtest.loc[entries_long, 'entry_price'],
               mode='markers', name='Long Entry', marker=dict(symbol='triangle-up', size=10, color='green')),
               row=candlestick_row, col=1)
fig_main.add_trace(go.Scatter(x=df_backtest.loc[entries_short, _KIEV_TIMESTAMP], y=df_backtest.loc[entries_short, 'entry_price'],
               mode='markers', name='Short Entry', marker=dict(symbol='triangle-down', size=10, color='red')),
               row=candlestick_row, col=1)
fig_main.add_trace(go.Scatter(x=df_backtest.loc[exits_tp, _KIEV_TIMESTAMP], y=df_backtest.loc[exits_tp, 'exit_price'],
               mode='markers', name='Take Profit', marker=dict(symbol='circle', size=10, color='lime', opacity=0.5, line=dict(width=1, color='darkgreen'))),
               row=candlestick_row, col=1)
fig_main.add_trace(go.Scatter(x=df_backtest.loc[exits_tp_trailing, _KIEV_TIMESTAMP], y=df_backtest.loc[exits_tp_trailing, 'exit_price'],
               mode='markers', name='TP (Trailing)', marker=dict(symbol='diamond', size=12, color='lightgreen', opacity=0.7, line=dict(width=1, color='green'))),
               row=candlestick_row, col=1)
fig_main.add_trace(go.Scatter(x=df_backtest.loc[exits_sl, _KIEV_TIMESTAMP], y=df_backtest.loc[exits_sl, 'exit_price'],
               mode='markers', name='Stop Loss', marker=dict(symbol='x', size=10, color='orange', line=dict(width=2, color='red'))),
               row=candlestick_row, col=1)
fig_main.add_trace(go.Scatter(x=df_backtest.loc[exits_sl_trailing, _KIEV_TIMESTAMP], y=df_backtest.loc[exits_sl_trailing, 'exit_price'],
               mode='markers', name='SL (Trailing)', marker=dict(symbol='diamond', size=12, color='gold', opacity=0.7, line=dict(width=2, color='darkorange'))),
               row=candlestick_row, col=1)

fig_main.show()
os.system('afplay /System/Library/Sounds/Glass.aiff')
print('\a')